# Can we generate marketplace-ready tent and backpack variants quickly enough for an interactive merchandising workflow?

## 1. Before You Begin

We will use the committed TrailMaster X4 Tent and Adventurer Pro Backpack
photographs as references. One edit request per product asks for two candidate
merchandising scenes. Our single focus is **multi-variant image editing** with
GPT-Image-2.5-Flare.

Deploy the model in Microsoft Foundry and configure section 2. Image generation
is a paid operation; this notebook does not call the service unless credentials
are present and the request cell is run. See
[current pricing](https://azure.microsoft.com/pricing/details/azure-openai/).


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_IMAGE_25_FLARE_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_GPT_IMAGE_25_FLARE_DEPLOYMENT"]
assets = find_assets()
output_dir = assets.parents[1] / "gpt-image-2.5-flare" / "output"
output_dir.mkdir(exist_ok=True)

print(f"Environment ready for deployment: {deployment}")
print(f"Generated images will be written to: {output_dir.resolve()}")


## 3. Inspect the approved source images

The notebook uses only the committed tent and backpack WebPs. The catalog
identifies them as product IDs 1 and 2; generated candidates go to `output/`
and never replace either source.


In [ ]:
# 4. Load both product records and display their committed photographs
import base64
import json

from IPython.display import HTML, Image, display

products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
selected = [product for product in products if product["id"] in {1, 2}]
if [product["id"] for product in selected] != [1, 2]:
    raise ValueError("Expected the committed product IDs 1 and 2")

def display_webp(path: Path, width: int = 360) -> None:
    """Render a local WebP through HTML because IPython Image cannot embed it."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/webp;base64,{encoded}" width="{width}">'))


for product in selected:
    source_image = assets / product["images"][0]
    if not source_image.is_file():
        raise FileNotFoundError(source_image)
    print(json.dumps({key: product[key] for key in ("id", "name", "brand", "category")}, indent=2))
    display_webp(source_image)


## 5. Request merchandising variants

One multipart edit request per product asks for two square candidates at low
quality. We record elapsed time for each request and make no claim about general
model latency. Each prompt asks the model to retain the product while varying
the retail setting.


In [ ]:
# 6. Edit both products into candidate retail scenes
import time

api_version = "2025-04-01-preview"
edit_url = (
    f"{endpoint}/openai/deployments/{deployment}/images/edits"
    f"?api-version={api_version}"
)
def create_variants(product: dict):
    """Create and save two variants for one approved product image."""
    source_image = assets / product["images"][0]
    prompt = (
        f"Create two marketplace-ready merchandising variants from this source "
        f"photo of {product['name']}. Keep the product recognizable and preserve "
        "its color and construction. Use clean outdoor-retail compositions with "
        "no added text or logos. Vary only the setting, mood, and lighting."
    )
    started = time.perf_counter()
    with source_image.open("rb") as image_file:
        response = requests.post(
            edit_url,
            headers={"api-key": api_key},
            data={
                "prompt": prompt,
                "n": "2",
                "size": "1024x1024",
                "quality": "low",
                "output_format": "png",
            },
            files={"image": (source_image.name, image_file, "image/webp")},
            timeout=240,
        )
    elapsed_seconds = time.perf_counter() - started
    if not response.ok:
        raise RuntimeError(f"Image edit failed ({response.status_code}): {response.text}")
    payload = response.json()
    if not payload.get("data"):
        raise RuntimeError(f"Image edit returned no image data: {payload}")

    output_paths = []
    for index, item in enumerate(payload["data"], start=1):
        image_bytes = base64.b64decode(item["b64_json"], validate=True)
        if not image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
            raise ValueError(f"Variant {index} for product {product['id']} is not a PNG")
        output_path = output_dir / f"flare-product-{product['id']}-variant-{index}.png"
        output_path.write_bytes(image_bytes)
        output_paths.append(output_path)
        display(Image(data=image_bytes, width=300))
    return output_paths, elapsed_seconds


for product in selected:
    output_paths, elapsed_seconds = create_variants(product)
    print(f"{product['name']}: saved {len(output_paths)} variants in {elapsed_seconds:.2f} seconds.")
print("These elapsed times describe these requests only; they are not a benchmark.")


## 7. Your Turn to Explore

- Request one variant for each product at `medium` quality.
- Ask for two settings that share the same lighting and camera angle.
- Tighten the prompt so all candidates share one background palette.


## 8. Summary

We used GPT-Image-2.5-Flare for one capability: creating several merchandising
variants from approved images with one edit request per product. We preserved
the sources, saved candidates under `output/`, and treated elapsed time as a local
observation. See the
[image-generation primer](../../../docs/primers/image-generation.md) for model
selection and review considerations.


## 9. References

- [GPT-Image-2.5-Flare model card](https://ai.azure.com/catalog/models/gpt-image-2.5-flare) — Foundry catalog entry.
- [Use image generation models from OpenAI](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/dall-e) — edits endpoint and response format.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — supported model version listing.
- [Create multimodal applications with OpenAI models in Microsoft Foundry](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/create-multimodal-applications-with-openai-models-in-microsoft-foundry/4543593) — release announcement.
